# s15 - Adversarial training sebagai defense: evaluasi R6 (SURVEY)

**Klaim R6**: adversarial training menaikkan robustness, dengan trade-off
[F16][F17][F32]. Kami bandingkan DUA model global:
- **clean** = dari run R3 (FedAvg biasa) -> `run_S_R3/global_final.npz`
- **adv-trained** = dari run R6 (client pakai `--adv-ratio`) -> `run_S_R6/global_final.npz`

Keduanya dievaluasi: (a) **clean MCC/acc/recall** (test bersih), (b) **robust MCC** di bawah
PGD (perturbasi di-clip ke persentil 1-99), (c) **asr** = proporsi serangan yang
terdeteksi sebelum diserang tetapi lolos sesudahnya.

**PROTOKOL (dikoreksi).** Model dilatih di EC2 dengan z-score LOKAL per klien, maka tiap
test set klien dinormalisasi dengan statistik train klien itu (batas clip juga dari train
klien itu). Dilaporkan per klien (`cic`, `unsw`) dan `union`.

**Prasyarat:** run R3 (clean) & R6 (adv) SELESAI di EC2.
**Output:** `survey_out/r6_adv_training.csv`.

In [ ]:
import importlib, sys, subprocess
for m in ('numpy','scikit-learn','boto3'):
    if importlib.util.find_spec(m.replace('scikit-learn','sklearn')) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
import os, io, numpy as np
from sklearn.metrics import matthews_corrcoef, accuracy_score
OUTDIR='survey_out'; os.makedirs(OUTDIR,exist_ok=True)
BUCKET='ssh-detection-features-232032302717'; REGION='ap-southeast-1'; BASE='unsw-far/federated_survey'
RUN_CLEAN=os.environ.get('RUN_CLEAN','S_R3'); RUN_ADV=os.environ.get('RUN_ADV','S_R6')
ADV_EPS=float(os.environ.get('ADV_EPS','0.1'))
import boto3; s3=boto3.client('s3',region_name=REGION)
def load_npz_s3(key):
    try: obj=s3.get_object(Bucket=BUCKET,Key=key)
    except Exception as e: print('MISS',key,e); return None
    return np.load(io.BytesIO(obj['Body'].read()),allow_pickle=True)

In [ ]:
P = {}
for nm in ['cic_train', 'cic_test', 'unsw_train', 'unsw_test']:
    sub = 'cic' if nm.startswith('cic') else 'unsw'
    d = load_npz_s3(f'{BASE}/data/{sub}/{nm}.npz'); assert d is not None, 'jalankan s01 dulu'
    P[nm] = (np.asarray(d['X'], float), np.asarray(d['y'], int))
CL = ['cic', 'unsw']
Z = {}   # per klien: z-space LOKAL (statistik dari TRAIN klien itu = sama dgn training FL di EC2)
for c in CL:
    Xtr_c = P[f'{c}_train'][0]
    m = Xtr_c.mean(0); s = Xtr_c.std(0); s[s == 0] = 1
    Ztr = (Xtr_c - m) / s
    Z[c] = dict(X=(P[f'{c}_test'][0] - m) / s, y=P[f'{c}_test'][1],
                lo=np.percentile(Ztr, 1, axis=0), hi=np.percentile(Ztr, 99, axis=0))
relu = lambda x: np.maximum(0, x); sig = lambda x: 1 / (1 + np.exp(-np.clip(x, -30, 30)))
def fwd(W, X):
    a1 = X @ W[0] + W[1]; z1 = relu(a1); a2 = z1 @ W[2] + W[3]; z2 = relu(a2); a3 = z2 @ W[4] + W[5]
    return sig(a3).ravel(), (a1, z1, a2, z2)
def dX(W, X, y):
    o, (a1, z1, a2, z2) = fwd(W, X); g = (o - y.astype(float))[:, None]
    d2 = (g @ W[4].T) * (a2 > 0); d1 = (d2 @ W[2].T) * (a1 > 0); return d1 @ W[0].T
def pred(W, X): o, _ = fwd(W, X); return (o >= 0.5).astype(int)
def pgd(W, X, y, eps, lo, hi, steps=10):
    a = eps / 4; Xa = X.copy()
    for _ in range(steps):
        Xa = np.clip(np.clip(Xa + a * np.sign(dX(W, Xa, y)), X - eps, X + eps), lo, hi)
    return Xa

## 1. Evaluasi clean MCC + robust MCC (PGD) utk tiap model

In [ ]:
from sklearn.metrics import recall_score

def evaluate(run):
    gf = load_npz_s3(f'{BASE}/run_{run}/global_final.npz')
    if gf is None: return None
    W = [gf[f'w{i}'] for i in range(6)]
    parts = {}
    for c in CL:
        X, y, lo, hi = Z[c]['X'], Z[c]['y'], Z[c]['lo'], Z[c]['hi']
        atk = (y == 1); p0 = pred(W, X)
        Xa = X.copy(); Xa[atk] = pgd(W, X[atk], y[atk], ADV_EPS, lo, hi); p1 = pred(W, Xa)
        parts[c] = (y, p0, p1, atk & (p0 == 1))
    parts['union'] = tuple(np.concatenate([parts[c][k] for c in CL]) for k in range(4))
    out = {}
    for scope, (y, p0, p1, det) in parts.items():
        out[scope] = dict(clean_mcc=float(matthews_corrcoef(y, p0)), clean_acc=float(accuracy_score(y, p0)),
                          clean_recall=float(recall_score(y, p0, zero_division=0)),
                          robust_mcc=float(matthews_corrcoef(y, p1)),
                          asr=float((p1[det] == 0).mean()) if det.sum() > 0 else float('nan'))
    return out

rows = []
for name, run in [('clean (FedAvg)', RUN_CLEAN), ('adv-trained', RUN_ADV)]:
    res = evaluate(run)
    if res is None:
        print(f'[TBD] {name}: global_final.npz run {run} belum ada'); continue
    for scope, m in res.items():
        rows.append(dict(model=name, scope=scope, **m))
    u = res['union']
    print(f'{name:16s} [union] clean_MCC={u["clean_mcc"]:+.3f} clean_acc={u["clean_acc"]:.3f} '
          f'clean_recall={u["clean_recall"]:.3f} robust_MCC(PGD)={u["robust_mcc"]:+.3f} ASR={u["asr"]:.3f}')
    for c in CL:
        v = res[c]; print(f'{"":16s} [{c}] clean_MCC={v["clean_mcc"]:+.3f} robust_MCC={v["robust_mcc"]:+.3f}')

## 2. Trade-off + simpan R6

In [ ]:
import csv
u = {r['model']: r for r in rows if r['scope'] == 'union'}
if len(u) == 2:
    c = u['clean (FedAvg)']; a = u['adv-trained']
    print(f'Robust gain (adv - clean) = {a["robust_mcc"] - c["robust_mcc"]:+.3f} MCC [union]')
    print(f'Clean cost  (adv - clean) = {a["clean_mcc"] - c["clean_mcc"]:+.3f} MCC, {a["clean_acc"] - c["clean_acc"]:+.3f} acc [union]')
path = os.path.join(OUTDIR, 'r6_adv_training.csv')
fields = ['model', 'scope', 'clean_mcc', 'clean_acc', 'clean_recall', 'robust_mcc', 'asr']
with open(path, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=fields); w.writeheader()
    for r in rows: w.writerow(r)
print('saved', path)
try: s3.upload_file(path, BUCKET, f'{BASE}/results/r6_adv_training.csv'); print('upload OK')
except Exception as e: print('(upload dilewati:', e, ')')
print('=== s15 SELESAI. ===')